In [1]:
import pandas as pd
import scanpy as sp
import anndata as ad
import numpy as np

In [2]:
adata_fat = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Fat.h5ad")
adata_liver = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Liver.h5ad")
adata_lung = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Lung.h5ad")
adata_marrow = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Marrow.h5ad")
adata_spleen = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Spleen.h5ad")
adata_kidney = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Kidney.h5ad")
adata_heart_and_aorta = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Heart_and_Aorta.h5ad")

C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\1967248991.py:1: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  adata_fat = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Fat.h5ad")
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\1967248991.py:1: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  adata_fat = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Fat.h5ad")
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\1967248991.py:2: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  adata_liver = ad.read_h5ad("../data_raw/tabula-muris-senis-droplet-processed-official-annotations-Liver.h5ad")
C:\Users\asd123cheese\AppD

In [3]:
from scipy.sparse import issparse

def check_counts(adata, name="str(adata)"):
    X = adata.X

    if issparse(X):
        nnz = X.nnz
        total = X.shape[0] * X.shape[1]
        sparsity = 1 - nnz / total
        data = X.data
    else:
        total = X.size
        flat = X.ravel()
        sparsity = (flat == 0).sum() / total
        data = flat

    # 核心检查
    is_non_negative = bool((data >= 0).all())
    is_integer = bool(np.allclose(data, np.round(data), atol=1e-5))
    min_val = float(data.min())
    max_val = float(data.max())
    mean_val = float(data[data > 0].mean()) if (data > 0).any() else 0

    if is_non_negative and is_integer:
        verdict = "✅ 原始 counts（非负整数）"
    elif is_non_negative:
        verdict = "⚠️ 非负但非整数 → normalized/log 数据"
    else:
        verdict = "❌ 含负值 → scaled 数据"

    print(f"{'='*50}")
    print(f"【{name}】 形状: {adata.shape}")
    print(f"  非零元素/总数:     {nnz:,} / {total:,}")
    print(f"  稀疏度:       {sparsity:.2%}")
    print(f"  最小值:       {min_val:.6f}")
    print(f"  最大值:       {max_val:.2f}")
    print(f"  非零均值:     {mean_val:.4f}")
    print(f"  非负:         {is_non_negative}")
    print(f"  近似整数:     {is_integer}")
    print(f"  判断:         {verdict}")
    print(f"{'='*50}")

datasets = [
    (adata_fat,              "fat"),
    (adata_liver,            "liver"),
    (adata_lung,             "lung"),
    (adata_marrow,           "marrow"),
    (adata_spleen,           "spleen"),
    (adata_kidney,           "kidney"),
    (adata_heart_and_aorta,  "heart_and_aorta"),
]

for adata, name in datasets:
    check_counts(adata, name)

【fat】 形状: (6777, 20138)
  非零元素/总数:     16,939,959 / 136,475,226
  稀疏度:       87.59%
  最小值:       0.054596
  最大值:       10.00
  非零均值:     2.3504
  非负:         True
  近似整数:     False
  判断:         ⚠️ 非负但非整数 → normalized/log 数据
【liver】 形状: (7294, 20138)
  非零元素/总数:     18,238,818 / 146,886,572
  稀疏度:       87.58%
  最小值:       0.033735
  最大值:       10.00
  非零均值:     2.2421
  非负:         True
  近似整数:     False
  判断:         ⚠️ 非负但非整数 → normalized/log 数据
【lung】 形状: (24540, 20138)
  非零元素/总数:     40,624,831 / 494,186,520
  稀疏度:       91.78%
  最小值:       0.102091
  最大值:       10.00
  非零均值:     2.8044
  非负:         True
  近似整数:     False
  判断:         ⚠️ 非负但非整数 → normalized/log 数据
【marrow】 形状: (40220, 20138)
  非零元素/总数:     77,595,548 / 809,950,360
  稀疏度:       90.42%
  最小值:       0.040045
  最大值:       10.00
  非零均值:     2.3452
  非负:         True
  近似整数:     False
  判断:         ⚠️ 非负但非整数 → normalized/log 数据
【spleen】 形状: (35718, 20138)
  非零元素/总数:     56,974,470 / 719,289,084
  稀疏度:       92.08%
  最小

In [4]:
raw_datasets = []
for adata, name in datasets:
    if adata.raw is None:
        print(f"⚠️ {name} 没有 .raw，跳过")
        continue
    raw_datasets.append((adata.raw, name+"_raw"))

for adata, name in raw_datasets:
    check_counts(adata, name)

【fat_raw】 形状: (6777, 20138)
  非零元素/总数:     16,939,959 / 136,475,226
  稀疏度:       87.59%
  最小值:       1.000000
  最大值:       39137.00
  非零均值:     4.4191
  非负:         True
  近似整数:     True
  判断:         ✅ 原始 counts（非负整数）
【liver_raw】 形状: (7294, 20138)
  非零元素/总数:     18,238,818 / 146,886,572
  稀疏度:       87.58%
  最小值:       1.000000
  最大值:       8406.00
  非零均值:     4.3529
  非负:         True
  近似整数:     True
  判断:         ✅ 原始 counts（非负整数）
【lung_raw】 形状: (24540, 20138)
  非零元素/总数:     40,624,831 / 494,186,520
  稀疏度:       91.78%
  最小值:       1.000000
  最大值:       5659.00
  非零均值:     3.0613
  非负:         True
  近似整数:     True
  判断:         ✅ 原始 counts（非负整数）
【marrow_raw】 形状: (40220, 20138)
  非零元素/总数:     77,595,548 / 809,950,360
  稀疏度:       90.42%
  最小值:       1.000000
  最大值:       13855.00
  非零均值:     5.4379
  非负:         True
  近似整数:     True
  判断:         ✅ 原始 counts（非负整数）
【spleen_raw】 形状: (35718, 20138)
  非零元素/总数:     56,974,470 / 719,289,084
  稀疏度:       92.08%
  最小值:       1.000000
  最大

In [5]:
for adata, name in raw_datasets:
    print(adata,name)

Raw AnnData with n_obs × n_vars = 6777 × 20138
    var: 'n_cells' fat_raw
Raw AnnData with n_obs × n_vars = 7294 × 20138
    var: 'n_cells' liver_raw
Raw AnnData with n_obs × n_vars = 24540 × 20138
    var: 'n_cells' lung_raw
Raw AnnData with n_obs × n_vars = 40220 × 20138
    var: 'n_cells' marrow_raw
Raw AnnData with n_obs × n_vars = 35718 × 20138
    var: 'n_cells' spleen_raw
Raw AnnData with n_obs × n_vars = 21647 × 20138
    var: 'n_cells' kidney_raw
Raw AnnData with n_obs × n_vars = 8613 × 20138
    var: 'n_cells' heart_and_aorta_raw


In [6]:
# 1. 从 raw_datasets 解包成 7 个独立变量 
(adata_fat_raw, _), \
(adata_liver_raw, _), \
(adata_lung_raw, _), \
(adata_marrow_raw, _), \
(adata_spleen_raw, _), \
(adata_kidney_raw, _), \
(adata_heart_and_aorta_raw, _) = raw_datasets

# 此时 adata_fat_raw 等都是 Raw 对象，可以 print 验证
print(adata_fat_raw)   # Raw object with n_obs × n_vars = ...

#  2. 转成 AnnData 并保存 h5ad 
save_names = [
    ("adata_fat_raw",            adata_fat_raw),
    ("adata_liver_raw",          adata_liver_raw),
    ("adata_lung_raw",           adata_lung_raw),
    ("adata_marrow_raw",         adata_marrow_raw),
    ("adata_spleen_raw",         adata_spleen_raw),
    ("adata_kidney_raw",         adata_kidney_raw),
    ("adata_heart_and_aorta_raw", adata_heart_and_aorta_raw),
]

import os

os.makedirs("../data_interim", exist_ok=True)  # 确保目录存在

for adata_orig, name in datasets:
    adata_save = adata_orig.raw.to_adata()
    adata_save.obs = adata_orig.obs.copy()
    filename = f"../data_interim/TMS_{name}.h5ad"   # ← 加上 ../data_interim/ 前缀
    adata_save.write_h5ad(filename)
    print(f"✅ 已保存: {filename}  shape={adata_save.shape}")

Raw AnnData with n_obs × n_vars = 6777 × 20138
    var: 'n_cells'
✅ 已保存: ../data_interim/TMS_fat.h5ad  shape=(6777, 20138)
✅ 已保存: ../data_interim/TMS_liver.h5ad  shape=(7294, 20138)
✅ 已保存: ../data_interim/TMS_lung.h5ad  shape=(24540, 20138)
✅ 已保存: ../data_interim/TMS_marrow.h5ad  shape=(40220, 20138)
✅ 已保存: ../data_interim/TMS_spleen.h5ad  shape=(35718, 20138)
✅ 已保存: ../data_interim/TMS_kidney.h5ad  shape=(21647, 20138)
✅ 已保存: ../data_interim/TMS_heart_and_aorta.h5ad  shape=(8613, 20138)


In [7]:
adata_fat_raw = ad.read_h5ad("../data_interim/TMS_fat.h5ad")
adata_liver_raw = ad.read_h5ad("../data_interim/TMS_liver.h5ad")
adata_lung_raw = ad.read_h5ad("../data_interim/TMS_lung.h5ad")
adata_marrow_raw = ad.read_h5ad("../data_interim/TMS_marrow.h5ad")
adata_spleen_raw = ad.read_h5ad("../data_interim/TMS_spleen.h5ad")
adata_kidney_raw = ad.read_h5ad("../data_interim/TMS_kidney.h5ad")
adata_heart_and_aorta_raw = ad.read_h5ad("../data_interim/TMS_heart_and_aorta.h5ad")

In [8]:
print(adata_fat_raw.obs.columns.tolist())

['age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden']


In [9]:
import hashlib
from pathlib import Path
from datetime import datetime

# 配置路径
DATA_DIR = Path("data/raw")          # 存放下载的h5ad文件
OUTPUT_DIR = Path("docs")            # 数据字典输出目录
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 需要检查的标准化字段
REQUIRED_FIELDS = ["mouse_id", "age_months", "sex", "tissue", "cell_type", "assay"]

In [10]:
# ============================================================
# Cell: 构建数据字典 — 逐字段检查
# ============================================================
from datetime import datetime
from pathlib import Path

# 所有已处理的 raw 数据集
datasets_raw = [
    (adata_fat_raw,              "fat"),
    (adata_liver_raw,            "liver"),
    (adata_lung_raw,             "lung"),
    (adata_marrow_raw,           "marrow"),
    (adata_spleen_raw,           "spleen"),
    (adata_kidney_raw,           "kidney"),
    (adata_heart_and_aorta_raw,  "heart_and_aorta"),
]

# 定义标准化字段 → 实际列名的映射（基于你已发现的 obs 列）
FIELD_MAPPING = {
    "mouse_id":    "mouse.id",
    "age_months":  "age",
    "sex":         "sex",
    "tissue":      "tissue",
    "cell_type":   "cell_ontology_class",   # 主注释
    "assay":       "method",
}

# 额外保留的辅助列
EXTRA_COLS = ["cell", "cell_ontology_id", "free_annotation", 
              "subtissue", "tissue_free_annotation", "n_genes", "n_counts"]

In [11]:
# ============================================================
# Cell: 逐文件、逐字段生成摘要
# ============================================================

def summarize_field(series, max_examples=5):
    """对一个 obs 列生成摘要字典"""
    n_total = len(series)
    n_missing = int(series.isna().sum())
    n_unique = series.nunique()
    examples = series.dropna().unique()[:max_examples].tolist()
    # 将 numpy 类型转为 python 原生类型，方便后续展示
    examples = [str(e) for e in examples]
    
    return {
        "dtype": str(series.dtype),
        "n_unique": n_unique,
        "n_missing": n_missing,
        "missing_pct": round(n_missing / n_total * 100, 2),
        "examples": examples,
    }

# 收集所有结果
all_summaries = {}  # {tissue_name: {std_field: summary_dict}}

for adata, tissue_name in datasets_raw:
    print(f"\n{'='*60}")
    print(f"  组织: {tissue_name}  |  细胞数: {adata.n_obs:,}  |  基因数: {adata.n_vars:,}")
    print(f"{'='*60}")
    
    tissue_summary = {}
    
    # --- 检查标准化字段 ---
    for std_name, actual_col in FIELD_MAPPING.items():
        if actual_col in adata.obs.columns:
            summary = summarize_field(adata.obs[actual_col])
            summary["actual_column"] = actual_col
            summary["status"] = "✅ 存在"
            tissue_summary[std_name] = summary
            print(f"  {std_name:12s} → {actual_col:25s} | "
                  f"唯一值={summary['n_unique']:>4d} | "
                  f"缺失={summary['missing_pct']:>5.1f}% | "
                  f"示例: {summary['examples'][:3]}")
        else:
            tissue_summary[std_name] = {
                "actual_column": None,
                "status": "❌ 缺失",
                "dtype": "N/A",
                "n_unique": 0,
                "n_missing": adata.n_obs,
                "missing_pct": 100.0,
                "examples": [],
            }
            print(f"  {std_name:12s} → ❌ 列 '{actual_col}' 不存在!")
    
    # --- 检查 counts 矩阵 ---
    X = adata.X
    from scipy.sparse import issparse
    if issparse(X):
        data_sample = X.data[:1000]
    else:
        data_sample = X.ravel()[:1000]
    
    is_non_neg = bool((data_sample >= 0).all())
    is_int = bool(np.allclose(data_sample, np.round(data_sample), atol=1e-5))
    
    tissue_summary["counts"] = {
        "location": ".X (已从 .raw.X 转换)",
        "is_raw_counts": is_non_neg and is_int,
        "matrix_dtype": str(X.dtype),
        "shape": str(X.shape),
        "status": "✅ 原始counts" if (is_non_neg and is_int) else "⚠️ 非原始counts",
    }
    print(f"  {'counts':12s} → .X | dtype={X.dtype} | "
          f"原始counts={'✅' if (is_non_neg and is_int) else '❌'}")
    
    all_summaries[tissue_name] = tissue_summary

print("\n\n✅ 所有组织字段检查完成！")


  组织: fat  |  细胞数: 6,777  |  基因数: 20,138
  mouse_id     → mouse.id                  | 唯一值=   6 | 缺失=  0.0% | 示例: ['18-F-50', '18-M-52', '18-M-53']
  age_months   → age                       | 唯一值=   3 | 缺失=  0.0% | 示例: ['18m', '21m', '30m']
  sex          → sex                       | 唯一值=   2 | 缺失=  0.0% | 示例: ['female', 'male']
  tissue       → tissue                    | 唯一值=   1 | 缺失=  0.0% | 示例: ['Fat']
  cell_type    → cell_ontology_class       | 唯一值=   8 | 缺失=  0.0% | 示例: ['endothelial cell', 'B cell', 'mesenchymal stem cell of adipose']
  assay        → method                    | 唯一值=   1 | 缺失=  0.0% | 示例: ['droplet']
  counts       → .X | dtype=float32 | 原始counts=✅

  组织: liver  |  细胞数: 7,294  |  基因数: 20,138
  mouse_id     → mouse.id                  | 唯一值=  12 | 缺失=  0.0% | 示例: ['18-F-51', '21-F-54', '24-M-58']
  age_months   → age                       | 唯一值=   6 | 缺失=  0.0% | 示例: ['18m', '21m', '24m']
  sex          → sex                       | 唯一值=   2 | 缺失=  0.0% | 示例:

In [12]:
# ============================================================
# Cell: 生成 metadata_mapping.tsv（原始列名 → 标准化字段名）
# ============================================================

mapping_records = []
for std_name, actual_col in FIELD_MAPPING.items():
    mapping_records.append({
        "standard_field": std_name,
        "original_column": actual_col,
        "location": "obs",
        "description": {
            "mouse_id": "独立小鼠个体编号，统计重复单位",
            "age_months": "小鼠月龄（数值型）",
            "sex": "性别 (male/female)",
            "tissue": "组织来源",
            "cell_type": "细胞类型注释（本体论分类）",
            "assay": "测序技术/方法 (如 10x Chromium)",
        }[std_name],
        "notes": {
            "mouse_id": "不可用细胞barcode代替；缺少则不可用于样本级统计",
            "age_months": "需后续建立 age_group (young/middle/old)",
            "sex": "pseudobulk模型中作为协变量",
            "tissue": "7个组织: fat/liver/lung/marrow/spleen/kidney/heart_and_aorta",
            "cell_type": "另有 cell(原始标签), free_annotation(自由注释) 可辅助",
            "assay": "可能与年龄混杂，模型中需作为批次协变量",
        }[std_name],
    })

# counts 单独一行
mapping_records.append({
    "standard_field": "counts",
    "original_column": ".raw.X → 已转为 .X",
    "location": ".X (AnnData主矩阵)",
    "description": "原始整数表达矩阵 (UMI counts)",
    "notes": "原始.X为log-normalized，已从.raw.X提取并覆盖保存",
})

df_mapping = pd.DataFrame(mapping_records)

os.makedirs("../docs", exist_ok=True)
df_mapping.to_csv("../docs/metadata_mapping.tsv", sep="\t", index=False)
print("✅ 已保存: ../docs/metadata_mapping.tsv")
df_mapping

✅ 已保存: ../docs/metadata_mapping.tsv


,standard_field,original_column,location,description,notes
0,mouse_id,mouse.id,obs,独立小鼠个体编号，统计重复单位,不可用细胞barcode代替；缺少则不可用于样本级统计
1,age_months,age,obs,小鼠月龄（数值型）,需后续建立 age_group (young/middle/old)
2,sex,sex,obs,性别 (male/female),pseudobulk模型中作为协变量
3,tissue,tissue,obs,组织来源,7个组织: fat/liver/lung/marrow/spleen/kidney/hear...
4,cell_type,cell_ontology_class,obs,细胞类型注释（本体论分类）,"另有 cell(原始标签), free_annotation(自由注释) 可辅助"
5,assay,method,obs,测序技术/方法 (如 10x Chromium),可能与年龄混杂，模型中需作为批次协变量
6,counts,.raw.X → 已转为 .X,.X (AnnData主矩阵),原始整数表达矩阵 (UMI counts),原始.X为log-normalized，已从.raw.X提取并覆盖保存


In [13]:
# ============================================================
# Cell: 生成 Markdown 数据字典
# ============================================================

def generate_markdown_dictionary(all_summaries, field_mapping, datasets_raw):
    lines = []
    lines.append("# 数据字典 (Metadata Dictionary)")
    lines.append(f"\n> **数据来源**: Tabula Muris Senis (TMS) - Droplet")
    lines.append(f"> **生成时间**: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    lines.append(f"> **组织数量**: {len(datasets_raw)}")
    lines.append(f"> **Counts来源**: `.raw.X`（原始 `.X` 为 log-normalized）\n")
    
    # ---- 1. 文件总览 ----
    lines.append("## 1. 文件总览\n")
    lines.append("| 组织 | 细胞数 | 基因数 | Counts位置 | 原始Counts | mouse_id |")
    lines.append("|------|--------|--------|-----------|-----------|----------|")
    
    for adata, name in datasets_raw:
        s = all_summaries[name]
        counts_ok = "✅" if s["counts"]["is_raw_counts"] else "❌"
        mouse_ok = "✅" if s["mouse_id"]["status"] == "✅ 存在" else "❌"
        lines.append(f"| {name} | {adata.n_obs:,} | {adata.n_vars:,} "
                     f"| {s['counts']['location']} | {counts_ok} | {mouse_ok} |")
    lines.append("")
    
    # ---- 2. 字段定义 ----
    lines.append("## 2. 字段定义与映射\n")
    lines.append("| 标准字段 | 原始列名 | 位置 | 含义 | 关键约束 |")
    lines.append("|----------|----------|------|------|----------|")
    
    constraints = {
        "mouse_id": "统计重复单位；缺少则不可用于样本级统计",
        "age_months": "需建立 age_group；分组边界第6周冻结",
        "sex": "pseudobulk模型协变量",
        "tissue": "需标准化为 tissue_std",
        "cell_type": "保留三列: cell / cell_ontology_class / free_annotation",
        "assay": "可能与年龄混杂，需作为批次协变量",
        "counts": "必须为原始整数counts，非log-normalized",
    }
    meanings = {
        "mouse_id": "独立小鼠个体编号",
        "age_months": "小鼠月龄",
        "sex": "性别",
        "tissue": "组织来源",
        "cell_type": "细胞类型（本体论分类）",
        "assay": "测序技术/方法",
        "counts": "原始UMI表达矩阵",
    }
    
    for std_name, actual_col in field_mapping.items():
        lines.append(f"| `{std_name}` | `{actual_col}` | `obs` "
                     f"| {meanings[std_name]} | {constraints[std_name]} |")
    lines.append(f"| `counts` | `.raw.X` → `.X` | `.X` "
                 f"| {meanings['counts']} | {constraints['counts']} |")
    lines.append("")
    
    # ---- 3. 逐组织字段详情 ----
    lines.append("## 3. 逐组织字段详情\n")
    
    for adata, tissue_name in datasets_raw:
        s = all_summaries[tissue_name]
        lines.append(f"### {tissue_name}\n")
        lines.append(f"- 细胞数: {adata.n_obs:,}")
        lines.append(f"- 基因数: {adata.n_vars:,}")
        lines.append(f"- Counts: {s['counts']['status']} (dtype={s['counts']['matrix_dtype']})\n")
        
        lines.append("| 标准字段 | 实际列名 | 类型 | 唯一值 | 缺失率 | 示例值 |")
        lines.append("|----------|----------|------|--------|--------|--------|")
        
        for std_name in field_mapping.keys():
            info = s[std_name]
            actual = info.get("actual_column", "N/A") or "❌"
            examples_str = ", ".join(info["examples"][:3]) if info["examples"] else "-"
            lines.append(f"| {std_name} | {actual} | {info['dtype']} "
                         f"| {info['n_unique']} | {info['missing_pct']}% "
                         f"| {examples_str} |")
        lines.append("")
    
    # ---- 4. 辅助列 ----
    lines.append("## 4. 辅助列（未标准化但保留）\n")
    lines.append("| 列名 | 说明 |")
    lines.append("|------|------|")
    lines.append("| `cell` | 原始细胞类型标签（比 cell_ontology_class 更细） |")
    lines.append("| `cell_ontology_id` | 细胞本体论 ID |")
    lines.append("| `free_annotation` | 自由注释（研究者手动标注） |")
    lines.append("| `subtissue` | 子组织区域 |")
    lines.append("| `tissue_free_annotation` | 组织自由注释 |")
    lines.append("| `n_genes` | 每个细胞检测到的基因数（QC指标） |")
    lines.append("| `n_counts` | 每个细胞的总UMI数（QC指标） |")
    lines.append("| `louvain` | Louvain聚类结果 |")
    lines.append("| `leiden` | Leiden聚类结果 |")
    lines.append("")
    
    # ---- 5. 关键发现与警告 ----
    lines.append("## 5. 关键发现\n")
    lines.append("1. **Counts位置**: 原始 `.X` 为 log-normalized 数据（非负但非整数），"
                 "原始 counts 存放在 `.raw.X` 中。已提取 `.raw.X` 覆盖保存至 `../data_interim/`。")
    lines.append("2. **mouse_id**: 所有组织均存在 `mouse.id` 列，可用于样本级统计。")
    lines.append("3. **age**: 以月龄（months）为单位，需后续离散化为 age_group。")
    lines.append("4. **cell_type**: 主注释使用 `cell_ontology_class`，"
                 "同时保留 `cell`（原始标签）和 `free_annotation`（自由注释）用于交叉验证。")
    lines.append("5. **assay/method**: 所有数据均为 droplet (10x Chromium)，"
                 "批次效应主要来自不同小鼠和不同组织。")
    lines.append("")
    
    return "\n".join(lines)


md_content = generate_markdown_dictionary(all_summaries, FIELD_MAPPING, datasets_raw)

# 保存
md_path = Path("../docs/metadata_dictionary.md")
md_path.write_text(md_content, encoding="utf-8")
print(f"✅ 数据字典已保存至: {md_path}")
print(f"\n{'='*60}")
print("预览前 2000 字符:")
print('='*60)
print(md_content[:2000])

✅ 数据字典已保存至: ..\docs\metadata_dictionary.md

预览前 2000 字符:
# 数据字典 (Metadata Dictionary)

> **数据来源**: Tabula Muris Senis (TMS) - Droplet
> **生成时间**: 2026-08-09 21:16
> **组织数量**: 7
> **Counts来源**: `.raw.X`（原始 `.X` 为 log-normalized）

## 1. 文件总览

| 组织 | 细胞数 | 基因数 | Counts位置 | 原始Counts | mouse_id |
|------|--------|--------|-----------|-----------|----------|
| fat | 6,777 | 20,138 | .X (已从 .raw.X 转换) | ✅ | ✅ |
| liver | 7,294 | 20,138 | .X (已从 .raw.X 转换) | ✅ | ✅ |
| lung | 24,540 | 20,138 | .X (已从 .raw.X 转换) | ✅ | ✅ |
| marrow | 40,220 | 20,138 | .X (已从 .raw.X 转换) | ✅ | ✅ |
| spleen | 35,718 | 20,138 | .X (已从 .raw.X 转换) | ✅ | ✅ |
| kidney | 21,647 | 20,138 | .X (已从 .raw.X 转换) | ✅ | ✅ |
| heart_and_aorta | 8,613 | 20,138 | .X (已从 .raw.X 转换) | ✅ | ✅ |

## 2. 字段定义与映射

| 标准字段 | 原始列名 | 位置 | 含义 | 关键约束 |
|----------|----------|------|------|----------|
| `mouse_id` | `mouse.id` | `obs` | 独立小鼠个体编号 | 统计重复单位；缺少则不可用于样本级统计 |
| `age_months` | `age` | `obs` | 小鼠月龄 | 需建立 age_group；分组边界第6周冻结 |
| `sex` | `se

In [14]:
# ============================================================
# Cell: 快速验证 — 检查 mouse_id 和 age 的分布（修正版）
# ============================================================

print("=" * 60)
print("  mouse.id × age × sex 交叉验证")
print("=" * 60)

for adata, name in datasets_raw:
    obs = adata.obs.copy()
    
    # 去掉 'm' 后缀，转为数值（月龄）
    obs["age_numeric"] = obs["age"].astype(str).str.replace("m", "", regex=False)
    obs["age_numeric"] = pd.to_numeric(obs["age_numeric"], errors="coerce")
    
    n_mice = obs["mouse.id"].nunique()
    age_min = obs["age_numeric"].min()
    age_max = obs["age_numeric"].max()
    age_range = f"{age_min:.0f} - {age_max:.0f} 月"
    sex_dist = obs["sex"].value_counts().to_dict()
    
    print(f"\n  [{name}]")
    print(f"    独立小鼠数: {n_mice}")
    print(f"    年龄范围:   {age_range}")
    print(f"    年龄唯一值: {sorted(obs['age_numeric'].dropna().unique().tolist())}")
    print(f"    性别分布:   {sex_dist}")
    
    # 每只小鼠的细胞数
    cells_per_mouse = obs.groupby("mouse.id").size()
    print(f"    每鼠细胞数: min={cells_per_mouse.min()}, "
          f"median={cells_per_mouse.median():.0f}, "
          f"max={cells_per_mouse.max()}")
    
    # 如果是 kidney，额外打印合并后的细胞类型分布
    if name == "kidney":
        print(f"    --- 合并后 kidney 细胞类型 TOP 10 ---")
        top10 = obs["cell_ontology_class"].value_counts().head(10)
        for ct, n in top10.items():
            print(f"      {ct}: {n}")

  mouse.id × age × sex 交叉验证

  [fat]
    独立小鼠数: 6
    年龄范围:   18 - 30 月
    年龄唯一值: [18, 21, 30]
    性别分布:   {'male': 4524, 'female': 2253}
    每鼠细胞数: min=378, median=951, max=2066

  [liver]
    独立小鼠数: 12
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 24, 30]
    性别分布:   {'male': 5663, 'female': 1631}
    每鼠细胞数: min=43, median=377, max=2320

  [lung]
    独立小鼠数: 16
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 30]
    性别分布:   {'male': 18736, 'female': 5804}
    每鼠细胞数: min=331, median=1221, max=8014

  [marrow]
    独立小鼠数: 17
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 24, 30]
    性别分布:   {'male': 28513, 'female': 11707}
    每鼠细胞数: min=1061, median=1962, max=7324

  [spleen]
    独立小鼠数: 13
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 24, 30]
    性别分布:   {'male': 18560, 'female': 17158}
    每鼠细胞数: min=1426, median=2943, max=4644

  [kidney]
    独立小鼠数: 16
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 24, 30]
    性别分布:   {'male': 16250, 'female': 5397}
    每鼠细胞数: min=524, median=1165, 

C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\1369426905.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cells_per_mouse = obs.groupby("mouse.id").size()
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\1369426905.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cells_per_mouse = obs.groupby("mouse.id").size()
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\1369426905.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future de

In [15]:
# ============================================================
# Cell: 统一将 age 列转为数值型（月龄）—— 只需运行一次
# ============================================================

for adata, name in datasets_raw:
    # '18m' → 18.0
    adata.obs["age_months"] = (
        adata.obs["age"]
        .astype(str)
        .str.replace("m", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )
    print(f"  [{name}] age_months: {sorted(adata.obs['age_months'].dropna().unique().tolist())}")

print("\n✅ 所有数据集已新增 'age_months' 数值列")

  [fat] age_months: [18, 21, 30]
  [liver] age_months: [1, 3, 18, 21, 24, 30]
  [lung] age_months: [1, 3, 18, 21, 30]
  [marrow] age_months: [1, 3, 18, 21, 24, 30]
  [spleen] age_months: [1, 3, 18, 21, 24, 30]
  [kidney] age_months: [1, 3, 18, 21, 24, 30]
  [heart_and_aorta] age_months: [1, 3, 18, 21, 24, 30]

✅ 所有数据集已新增 'age_months' 数值列


In [16]:
# ============================================================
# Cell: 生成 sample_coverage.csv（组织 × 年龄 × 小鼠 覆盖表）
# ============================================================

coverage_records = []

for adata, name in datasets_raw:
    obs = adata.obs
    grouped = obs.groupby(["mouse.id", "age", "sex"]).size().reset_index(name="n_cells")
    grouped["tissue"] = name
    coverage_records.append(grouped)

df_coverage = pd.concat(coverage_records, ignore_index=True)
df_coverage = df_coverage[["tissue", "mouse.id", "age", "sex", "n_cells"]]
df_coverage.columns = ["tissue", "mouse_id", "age_months", "sex", "n_cells"]

# 保存
df_coverage.to_csv("../docs/sample_coverage.csv", index=False)
print(f"✅ 已保存: ../docs/sample_coverage.csv")
print(f"   共 {len(df_coverage)} 条记录（组织×小鼠×年龄×性别）")
print(f"\n前 15 行预览:")
df_coverage.head(15)

C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\3230908034.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = obs.groupby(["mouse.id", "age", "sex"]).size().reset_index(name="n_cells")
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\3230908034.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = obs.groupby(["mouse.id", "age", "sex"]).size().reset_index(name="n_cells")
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\3230908034.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fal

✅ 已保存: ../docs/sample_coverage.csv
   共 1024 条记录（组织×小鼠×年龄×性别）

前 15 行预览:


C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\3230908034.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = obs.groupby(["mouse.id", "age", "sex"]).size().reset_index(name="n_cells")
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\3230908034.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = obs.groupby(["mouse.id", "age", "sex"]).size().reset_index(name="n_cells")
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\3230908034.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fal

,tissue,mouse_id,age_months,sex,n_cells
0,fat,18-F-50,18m,female,1459
1,fat,18-F-50,18m,male,0
2,fat,18-F-50,21m,female,0
3,fat,18-F-50,21m,male,0
4,fat,18-F-50,30m,female,0
5,fat,18-F-50,30m,male,0
6,fat,18-M-52,18m,female,0
7,fat,18-M-52,18m,male,2066
8,fat,18-M-52,21m,female,0
9,fat,18-M-52,21m,male,0


In [17]:
# ============================================================
# Cell: 生成 celltype_coverage.csv（组织 × 细胞类型 覆盖矩阵）
# ============================================================

celltype_records = []

for adata, name in datasets_raw:
    obs = adata.obs
    # 每个细胞类型有多少独立小鼠
    ct_mice = obs.groupby("cell_ontology_class")["mouse.id"].nunique().reset_index()
    ct_mice.columns = ["cell_type", "n_mice"]
    # 每个细胞类型有多少细胞
    ct_cells = obs["cell_ontology_class"].value_counts().reset_index()
    ct_cells.columns = ["cell_type", "n_cells"]
    
    ct_summary = ct_mice.merge(ct_cells, on="cell_type")
    ct_summary["tissue"] = name
    celltype_records.append(ct_summary)

df_celltype = pd.concat(celltype_records, ignore_index=True)
df_celltype = df_celltype[["tissue", "cell_type", "n_mice", "n_cells"]]
df_celltype = df_celltype.sort_values(["tissue", "n_cells"], ascending=[True, False])

df_celltype.to_csv("../docs/celltype_coverage.csv", index=False)
print(f"✅ 已保存: ../docs/celltype_coverage.csv")
print(f"   共 {len(df_celltype)} 条记录")
print(f"\n各组织细胞类型数量:")
print(df_celltype.groupby("tissue")["cell_type"].count())

✅ 已保存: ../docs/celltype_coverage.csv
   共 109 条记录

各组织细胞类型数量:
tissue
fat                 8
heart_and_aorta     9
kidney             23
liver               9
lung               30
marrow             18
spleen             12
Name: cell_type, dtype: int64


C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\951771944.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ct_mice = obs.groupby("cell_ontology_class")["mouse.id"].nunique().reset_index()
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\951771944.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ct_mice = obs.groupby("cell_ontology_class")["mouse.id"].nunique().reset_index()
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\951771944.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to re

In [18]:

# ============================================================
# Cell: 按纳入/排除标准进行层级筛选（mouse / tissue / cell type / expression / external）
# ============================================================

import pandas as pd
import numpy as np
from scipy.sparse import issparse

FILTER_LOG = []

def record_filter(level, item, status, reason):
    FILTER_LOG.append({
        "level": level,
        "item": item,
        "status": status,
        "reason": reason
    })


def hierarchical_filter(adata, tissue_name):
    """
    根据预设标准：
    1. mouse_id明确；年龄、性别可用；同组织至少2个年龄层
    2. 目标细胞类型需要多个年龄层、多只小鼠支持
    3. 细胞类型需要映射统一上位类别
    4. 保留原始整数counts
    """
    obs = adata.obs.copy()

    # ---------------------------
    # 1. Mouse层级检查
    # ---------------------------
    required_mouse = ["mouse.id", "age_months"]
    missing = [x for x in required_mouse if x not in obs.columns]

    if missing:
        record_filter("mouse", tissue_name, "降级", 
                      f"缺少字段: {missing}")
        return None

    mouse_num = obs["mouse.id"].nunique()
    age_num = obs["age_months"].nunique()

    if mouse_num < 2:
        record_filter("mouse", tissue_name, "降级",
                      "独立小鼠数量不足")
        return None

    if age_num < 2:
        record_filter("mouse", tissue_name, "降级",
                      "单一年龄层")
        return None

    record_filter("mouse", tissue_name, "通过",
                  f"{mouse_num}只小鼠，{age_num}个年龄层")


    # ---------------------------
    # 2. Tissue层级检查
    # ---------------------------
    ct_col = "cell_ontology_class"

    if ct_col not in obs.columns:
        record_filter("tissue", tissue_name, "降级",
                      "缺少细胞类型注释")
        return None

    ct_support = (
        obs.groupby(ct_col)["mouse.id"]
        .nunique()
    )

    valid_ct = ct_support[ct_support >= 3].index

    if len(valid_ct) == 0:
        record_filter("tissue", tissue_name, "降级",
                      "没有由至少3只小鼠支持的细胞类型")
        return None

    adata = adata[obs[ct_col].isin(valid_ct)].copy()

    record_filter("tissue", tissue_name, "通过",
                  f"保留{len(valid_ct)}个细胞类型")


    # ---------------------------
    # 3. 表达矩阵检查
    # ---------------------------
    X = adata.X

    if issparse(X):
        values = X.data
    else:
        values = X.flatten()

    # 检查是否为整数counts
    if not np.all(np.equal(np.mod(values, 1), 0)):
        record_filter("expression", tissue_name, "降级",
                      "表达矩阵不是整数counts")
    else:
        record_filter("expression", tissue_name, "通过",
                      "保留原始counts")


    return adata


filtered_dataset = []

for adata, name in datasets_raw:
    print("="*60)
    print("检查:", name)

    result = hierarchical_filter(adata, name)

    if result is not None:
        filtered_dataset.append((result, name))


filter_report = pd.DataFrame(FILTER_LOG)
filter_report.to_csv(
    "../docs/hierarchical_filter_report.csv",
    index=False
)

print("\n筛选完成")
display(filter_report)


检查: fat


C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_3196\3119712656.py:69: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs.groupby(ct_col)["mouse.id"]


检查: liver
检查: lung
检查: marrow
检查: spleen
检查: kidney
检查: heart_and_aorta

筛选完成


,level,item,status,reason
0,mouse,fat,通过,6只小鼠，3个年龄层
1,tissue,fat,通过,保留8个细胞类型
2,expression,fat,通过,保留原始counts
3,mouse,liver,通过,12只小鼠，6个年龄层
4,tissue,liver,通过,保留8个细胞类型
5,expression,liver,通过,保留原始counts
6,mouse,lung,通过,16只小鼠，5个年龄层
7,tissue,lung,通过,保留29个细胞类型
8,expression,lung,通过,保留原始counts
9,mouse,marrow,通过,17只小鼠，6个年龄层


In [19]:
# ============================================================
# 0. 配置
# ============================================================
data_dir = "../data_interim/"
tissues = ["Fat", "Liver", "Lung", "Marrow", "Spleen", "Kidney", "Heart_and_Aorta"]

# 目标细胞类型关键词（基于 cell_ontology_class 列）
target_keywords = {
    "endothelial": ["endothelial", "endothelial cell"],
    "macrophage/monocyte": ["macrophage", "monocyte", "classical monocyte", 
                            "non-classical monocyte", "intermediate monocyte"],
    "fibroblast/stromal": ["fibroblast", "stromal cell", "mesenchymal stem cell",
                           "pericyte", "smooth muscle cell"],
}

cell_type_col = "cell_ontology_class"


In [20]:
# ============================================================
# 1. 读取所有组织数据并检查细胞类型
# ============================================================
print("=" * 60)
print("Step 1: 读取数据并检查细胞类型")
print("=" * 60)

adata_dict = {}
all_cell_types = set()

for tissue in tissues:
    fpath = os.path.join(data_dir, f"TMS_{tissue}.h5ad")
    if not os.path.exists(fpath):
        print(f"  ⚠️ 文件不存在: {fpath}")
        continue
    adata = sp.read_h5ad(fpath)
    adata_dict[tissue] = adata
    cell_types = adata.obs[cell_type_col].unique().tolist()
    all_cell_types.update(cell_types)
    print(f"  {tissue}: {adata.n_obs} cells, {len(cell_types)} cell types")

# 打印所有唯一细胞类型
print(f"\n所有组织中出现的细胞类型 ({len(all_cell_types)} 种):")
for ct in sorted(all_cell_types):
    print(f"  - {ct}")

Step 1: 读取数据并检查细胞类型
  Fat: 6777 cells, 8 cell types
  Liver: 7294 cells, 9 cell types
  Lung: 24540 cells, 30 cell types
  Marrow: 40220 cells, 18 cell types
  Spleen: 35718 cells, 12 cell types
  Kidney: 21647 cells, 23 cell types
  Heart_and_Aorta: 8613 cells, 9 cell types

所有组织中出现的细胞类型 (83 种):
  - B cell
  - CD4-positive, alpha-beta T cell
  - CD8-positive, alpha-beta T cell
  - Kupffer cell
  - NK cell
  - T cell
  - adventitial cell
  - alveolar macrophage
  - basophil
  - bronchial smooth muscle cell
  - brush cell
  - cardiac neuron
  - cardiomyocyte
  - ciliated columnar cell of tracheobronchial tree
  - classical monocyte
  - club cell of bronchiole
  - dendritic cell
  - duct epithelial cell
  - endocardial cell
  - endothelial cell
  - endothelial cell of coronary artery
  - endothelial cell of hepatic sinusoid
  - endothelial cell of lymphatic vessel
  - epithelial cell
  - epithelial cell of proximal tubule
  - erythroblast
  - erythrocyte
  - erythroid progenitor
  - fene

In [21]:
# ============================================================
# 3. 合并 Spleen 和 Marrow（不过滤细胞种类）
# ============================================================
print("\n" + "=" * 60)
print("Step 3: 合并 Spleen 和 Marrow")
print("=" * 60)

merge_tissues = ["Spleen", "Marrow"]
other_tissues = [t for t in adata_dict.keys() if t not in merge_tissues]

# 合并 Spleen + Marrow
merge_list = []
for t in merge_tissues:
    if t in adata_dict:
        tmp = adata_dict[t].copy()
        tmp.obs["tissue"] = "spleen/marrow"
        merge_list.append(tmp)
        print(f"  {t} → spleen/marrow: {tmp.n_obs} cells")

if merge_list:
    adata_spleen_marrow = sp.concat(merge_list, join="outer")
    print(f"\n  合并后 spleen/marrow: {adata_spleen_marrow.n_obs} cells")
else:
    adata_spleen_marrow = None
    print("  未找到 Spleen/Marrow 数据，跳过合并")


Step 3: 合并 Spleen 和 Marrow
  Spleen → spleen/marrow: 35718 cells
  Marrow → spleen/marrow: 40220 cells

  合并后 spleen/marrow: 75938 cells


In [22]:
# ============================================================
# 4. 合并所有组织为最终 AnnData（不过滤细胞种类）
# ============================================================
print("\n" + "=" * 60)
print("Step 4: 合并所有组织为最终文件")
print("=" * 60)

final_list = []

# 加入 spleen/marrow 合并数据
if adata_spleen_marrow is not None:
    final_list.append(adata_spleen_marrow)
    print(f"  spleen/marrow: {adata_spleen_marrow.n_obs} cells")

# 加入其他组织（直接用 adata_dict，不做细胞类型过滤）
for t in other_tissues:
    final_list.append(adata_dict[t])
    print(f"  {t}: {adata_dict[t].n_obs} cells")

adata_final = sp.concat(final_list, join="outer")

# 确保格式与原始一致：.X 为原始 counts
print(f"\n  最终数据: {adata_final.n_obs} cells × {adata_final.n_vars} genes")
print(f"  .X dtype: {adata_final.X.dtype}")
print(f"  .X 是否为整数: {np.allclose(adata_final.X.data, np.round(adata_final.X.data))}")


Step 4: 合并所有组织为最终文件
  spleen/marrow: 75938 cells
  Fat: 6777 cells
  Liver: 7294 cells
  Lung: 24540 cells
  Kidney: 21647 cells
  Heart_and_Aorta: 8613 cells

  最终数据: 144809 cells × 20138 genes
  .X dtype: float32
  .X 是否为整数: True


In [23]:
# ============================================================
# 5. 保存
# ============================================================
print("\n" + "=" * 60)
print("Step 5: 保存结果")
print("=" * 60)

output_path = "../data_interim/TMS_checked.h5ad"
adata_final.write_h5ad(output_path)
print(f"  ✅ 已保存: {output_path}")


Step 5: 保存结果
  ✅ 已保存: ../data_interim/TMS_checked.h5ad


In [24]:
# ============================================================
# 6. 验证与汇总
# ============================================================
print("\n" + "=" * 60)
print("Step 6: 最终数据验证")
print("=" * 60)

print(f"\n组织分布:")
print(adata_final.obs["tissue"].value_counts().to_string())

print(f"\n细胞类型分布:")
print(adata_final.obs[cell_type_col].value_counts().to_string())

print(f"\nobs 列名: {list(adata_final.obs.columns)}")
print(f"\n前5行 obs:")
print(adata_final.obs.head())


Step 6: 最终数据验证

组织分布:
tissue
spleen/marrow      75938
Lung               24540
Kidney             21647
Heart_and_Aorta     8613
Liver               7294
Fat                 6777

细胞类型分布:
cell_ontology_class
B cell                                                       27577
granulocyte                                                  12211
classical monocyte                                            7922
granulocytopoietic cell                                       7599
T cell                                                        6880
proerythroblast                                               4570
kidney proximal convoluted tubule epithelial cell             4460
macrophage                                                    3823
endothelial cell of coronary artery                           3293
epithelial cell of proximal tubule                            3053
mature NK T cell                                              2962
hepatocyte                                            

In [25]:
# ============================================================
# Cell: 加载筛选后的整合文件，构建 datasets_raw
# ============================================================
adata_final = sp.read_h5ad("../data_interim/TMS_endo_macro_fibro.h5ad")
print(f"已加载: {adata_final.n_obs:,} cells × {adata_final.n_vars:,} genes")
print(f"组织: {adata_final.obs['tissue'].unique().tolist()}")

# 按 tissue 拆分，构建与原代码相同格式的 datasets_raw
datasets_raw = [
    (adata_final[adata_final.obs["tissue"] == t].copy(), t)
    for t in adata_final.obs["tissue"].unique()
]

for adata, name in datasets_raw:
    print(f"  {name}: {adata.n_obs:,} cells")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '../data_interim/TMS_endo_macro_fibro.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# ============================================================
# Cell: 逐组织字段检查（复用 summarize_field）
# ============================================================
all_summaries = {}

for adata, tissue_name in datasets_raw:
    print(f"\n{'='*60}")
    print(f"  组织: {tissue_name}  |  细胞数: {adata.n_obs:,}  |  基因数: {adata.n_vars:,}")
    print(f"{'='*60}")
    
    tissue_summary = {}
    
    for std_name, actual_col in FIELD_MAPPING.items():
        if actual_col in adata.obs.columns:
            summary = summarize_field(adata.obs[actual_col])
            summary["actual_column"] = actual_col
            summary["status"] = "✅ 存在"
            tissue_summary[std_name] = summary
            print(f"  {std_name:12s} → {actual_col:25s} | "
                  f"唯一值={summary['n_unique']:>4d} | "
                  f"缺失={summary['missing_pct']:>5.1f}% | "
                  f"示例: {summary['examples'][:3]}")
        else:
            tissue_summary[std_name] = {
                "actual_column": None,
                "status": "❌ 缺失",
                "dtype": "N/A",
                "n_unique": 0,
                "n_missing": adata.n_obs,
                "missing_pct": 100.0,
                "examples": [],
            }
            print(f"  {std_name:12s} → ❌ 列 '{actual_col}' 不存在!")
    
    # counts 检查
    X = adata.X
    if issparse(X):
        data_sample = X.data[:1000]
    else:
        data_sample = X.ravel()[:1000]
    
    is_non_neg = bool((data_sample >= 0).all())
    is_int = bool(np.allclose(data_sample, np.round(data_sample), atol=1e-5))
    
    tissue_summary["counts"] = {
        "location": ".X (原始counts)",
        "is_raw_counts": is_non_neg and is_int,
        "matrix_dtype": str(X.dtype),
        "shape": str(X.shape),
        "status": "✅ 原始counts" if (is_non_neg and is_int) else "⚠️ 非原始counts",
    }
    print(f"  {'counts':12s} → .X | dtype={X.dtype} | "
          f"原始counts={'✅' if (is_non_neg and is_int) else '❌'}")
    
    all_summaries[tissue_name] = tissue_summary

print("\n✅ 所有组织字段检查完成！")


  组织: spleen/marrow  |  细胞数: 7,784  |  基因数: 20,138
  mouse_id     → mouse.id                  | 唯一值=  19 | 缺失=  0.0% | 示例: ['18-F-50', '18-F-51', '21-F-54']
  age_months   → age                       | 唯一值=   6 | 缺失=  0.0% | 示例: ['18m', '21m', '24m']
  sex          → sex                       | 唯一值=   2 | 缺失=  0.0% | 示例: ['female', 'male']
  tissue       → tissue                    | 唯一值=   1 | 缺失=  0.0% | 示例: ['spleen/marrow']
  cell_type    → cell_ontology_class       | 唯一值=   4 | 缺失=  0.0% | 示例: ['macrophage', 'macrophage dendritic cell progenitor', 'monocyte']
  assay        → method                    | 唯一值=   1 | 缺失=  0.0% | 示例: ['droplet']
  counts       → .X | dtype=float32 | 原始counts=✅

  组织: Fat  |  细胞数: 3,368  |  基因数: 20,138
  mouse_id     → mouse.id                  | 唯一值=   6 | 缺失=  0.0% | 示例: ['18-F-50', '18-M-52', '18-M-53']
  age_months   → age                       | 唯一值=   3 | 缺失=  0.0% | 示例: ['18m', '21m', '30m']
  sex          → sex                       | 唯一值=   2

In [ ]:
# ============================================================
# Cell: 生成 metadata_mapping_filtered.tsv
# ============================================================
mapping_records = []
for std_name, actual_col in FIELD_MAPPING.items():
    mapping_records.append({
        "standard_field": std_name,
        "original_column": actual_col,
        "location": "obs",
        "description": {
            "mouse_id": "独立小鼠个体编号，统计重复单位",
            "age_months": "小鼠月龄（数值型）",
            "sex": "性别 (male/female)",
            "tissue": "组织来源（spleen/marrow已合并）",
            "cell_type": "细胞类型注释（仅内皮/巨噬单核/成纤维基质）",
            "assay": "测序技术/方法 (如 10x Chromium)",
        }[std_name],
        "notes": {
            "mouse_id": "不可用细胞barcode代替；缺少则不可用于样本级统计",
            "age_months": "需后续建立 age_group (young/middle/old)",
            "sex": "pseudobulk模型中作为协变量",
            "tissue": "6个组织: fat/liver/lung/kidney/heart_and_aorta/spleen_marrow",
            "cell_type": "仅保留内皮/巨噬单核/成纤维基质三大类",
            "assay": "可能与年龄混杂，模型中需作为批次协变量",
        }[std_name],
    })

mapping_records.append({
    "standard_field": "counts",
    "original_column": ".raw.X → 已转为 .X",
    "location": ".X (AnnData主矩阵)",
    "description": "原始整数表达矩阵 (UMI counts)",
    "notes": "原始.X为log-normalized，已从.raw.X提取并覆盖保存",
})

df_mapping = pd.DataFrame(mapping_records)
df_mapping.to_csv("../docs/metadata_mapping_filtered.tsv", sep="\t", index=False)
print("✅ 已保存: ../docs/metadata_mapping_filtered.tsv")
df_mapping

✅ 已保存: ../docs/metadata_mapping_filtered.tsv


,standard_field,original_column,location,description,notes
0,mouse_id,mouse.id,obs,独立小鼠个体编号，统计重复单位,不可用细胞barcode代替；缺少则不可用于样本级统计
1,age_months,age,obs,小鼠月龄（数值型）,需后续建立 age_group (young/middle/old)
2,sex,sex,obs,性别 (male/female),pseudobulk模型中作为协变量
3,tissue,tissue,obs,组织来源（spleen/marrow已合并）,6个组织: fat/liver/lung/kidney/heart_and_aorta/sp...
4,cell_type,cell_ontology_class,obs,细胞类型注释（仅内皮/巨噬单核/成纤维基质）,仅保留内皮/巨噬单核/成纤维基质三大类
5,assay,method,obs,测序技术/方法 (如 10x Chromium),可能与年龄混杂，模型中需作为批次协变量
6,counts,.raw.X → 已转为 .X,.X (AnnData主矩阵),原始整数表达矩阵 (UMI counts),原始.X为log-normalized，已从.raw.X提取并覆盖保存


In [ ]:
# ============================================================
# Cell: 生成 Markdown 数据字典
# ============================================================
md_content = generate_markdown_dictionary(all_summaries, FIELD_MAPPING, datasets_raw)

md_path = Path("../docs/metadata_dictionary_filtered.md")
md_path.write_text(md_content, encoding="utf-8")
print(f"✅ 数据字典已保存: {md_path}")
print(f"\n预览前 2000 字符:")
print("=" * 60)
print(md_content[:2000])

✅ 数据字典已保存: ..\docs\metadata_dictionary_filtered.md

预览前 2000 字符:
# 数据字典 (Metadata Dictionary)

> **数据来源**: Tabula Muris Senis (TMS) - Droplet
> **生成时间**: 2026-08-09 20:06
> **组织数量**: 6
> **Counts来源**: `.raw.X`（原始 `.X` 为 log-normalized）

## 1. 文件总览

| 组织 | 细胞数 | 基因数 | Counts位置 | 原始Counts | mouse_id |
|------|--------|--------|-----------|-----------|----------|
| spleen/marrow | 7,784 | 20,138 | .X (原始counts) | ✅ | ✅ |
| Fat | 3,368 | 20,138 | .X (原始counts) | ✅ | ✅ |
| Liver | 3,260 | 20,138 | .X (原始counts) | ✅ | ✅ |
| Lung | 18,334 | 20,138 | .X (原始counts) | ✅ | ✅ |
| Kidney | 1,729 | 20,138 | .X (原始counts) | ✅ | ✅ |
| Heart_and_Aorta | 6,138 | 20,138 | .X (原始counts) | ✅ | ✅ |

## 2. 字段定义与映射

| 标准字段 | 原始列名 | 位置 | 含义 | 关键约束 |
|----------|----------|------|------|----------|
| `mouse_id` | `mouse.id` | `obs` | 独立小鼠个体编号 | 统计重复单位；缺少则不可用于样本级统计 |
| `age_months` | `age` | `obs` | 小鼠月龄 | 需建立 age_group；分组边界第6周冻结 |
| `sex` | `sex` | `obs` | 性别 | pseudobulk模型协变量 |
| `tissue` | `tissue` | `obs` | 

In [ ]:
# ============================================================
# Cell: mouse.id × age × sex 交叉验证
# ============================================================
print("=" * 60)
print("  mouse.id × age × sex 交叉验证")
print("=" * 60)

for adata, name in datasets_raw:
    obs = adata.obs.copy()
    
    obs["age_numeric"] = obs["age"].astype(str).str.replace("m", "", regex=False)
    obs["age_numeric"] = pd.to_numeric(obs["age_numeric"], errors="coerce")
    
    n_mice = obs["mouse.id"].nunique()
    age_min = obs["age_numeric"].min()
    age_max = obs["age_numeric"].max()
    sex_dist = obs["sex"].value_counts().to_dict()
    
    print(f"\n  [{name}]")
    print(f"    独立小鼠数: {n_mice}")
    print(f"    年龄范围:   {age_min:.0f} - {age_max:.0f} 月")
    print(f"    年龄唯一值: {sorted(obs['age_numeric'].dropna().unique().tolist())}")
    print(f"    性别分布:   {sex_dist}")
    
    cells_per_mouse = obs.groupby("mouse.id").size()
    print(f"    每鼠细胞数: min={cells_per_mouse.min()}, "
          f"median={cells_per_mouse.median():.0f}, "
          f"max={cells_per_mouse.max()}")

  mouse.id × age × sex 交叉验证

  [spleen/marrow]
    独立小鼠数: 19
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 24, 30]
    性别分布:   {'male': 4850, 'female': 2934}
    每鼠细胞数: min=106, median=399, max=1211

  [Fat]
    独立小鼠数: 6
    年龄范围:   18 - 30 月
    年龄唯一值: [18, 21, 30]
    性别分布:   {'male': 2319, 'female': 1049}
    每鼠细胞数: min=197, median=430, max=1310

  [Liver]
    独立小鼠数: 12
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 24, 30]
    性别分布:   {'male': 2941, 'female': 319}
    每鼠细胞数: min=2, median=28, max=1611

  [Lung]
    独立小鼠数: 16
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 30]
    性别分布:   {'male': 15281, 'female': 3053}
    每鼠细胞数: min=159, median=746, max=7963

  [Kidney]
    独立小鼠数: 16
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 24, 30]
    性别分布:   {'male': 1330, 'female': 399}
    每鼠细胞数: min=45, median=86, max=247

  [Heart_and_Aorta]
    独立小鼠数: 11
    年龄范围:   1 - 30 月
    年龄唯一值: [1, 3, 18, 21, 24, 30]
    性别分布:   {'male': 4813, 'female': 1325}
    每鼠细胞数: min=114, median=278, ma

C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\298720791.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cells_per_mouse = obs.groupby("mouse.id").size()
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\298720791.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cells_per_mouse = obs.groupby("mouse.id").size()
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\298720791.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future de

In [ ]:
# ============================================================
# Cell: 生成 sample_coverage_filtered.csv
# ============================================================
coverage_records = []

for adata, name in datasets_raw:
    obs = adata.obs
    grouped = obs.groupby(["mouse.id", "age", "sex"]).size().reset_index(name="n_cells")
    grouped["tissue"] = name
    coverage_records.append(grouped)

df_coverage = pd.concat(coverage_records, ignore_index=True)
df_coverage = df_coverage[["tissue", "mouse.id", "age", "sex", "n_cells"]]
df_coverage.columns = ["tissue", "mouse_id", "age_months", "sex", "n_cells"]

df_coverage.to_csv("../docs/sample_coverage_filtered.csv", index=False)
print(f"✅ 已保存: ../docs/sample_coverage_filtered.csv")
print(f"   共 {len(df_coverage)} 条记录")
print(f"\n前 15 行:")
df_coverage.head(15)

✅ 已保存: ../docs/sample_coverage_filtered.csv
   共 892 条记录

前 15 行:


C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\2413967874.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = obs.groupby(["mouse.id", "age", "sex"]).size().reset_index(name="n_cells")
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\2413967874.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = obs.groupby(["mouse.id", "age", "sex"]).size().reset_index(name="n_cells")
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\2413967874.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=

,tissue,mouse_id,age_months,sex,n_cells
0,spleen/marrow,1-M-62,1m,female,0
1,spleen/marrow,1-M-62,1m,male,276
2,spleen/marrow,1-M-62,3m,female,0
3,spleen/marrow,1-M-62,3m,male,0
4,spleen/marrow,1-M-62,18m,female,0
5,spleen/marrow,1-M-62,18m,male,0
6,spleen/marrow,1-M-62,21m,female,0
7,spleen/marrow,1-M-62,21m,male,0
8,spleen/marrow,1-M-62,24m,female,0
9,spleen/marrow,1-M-62,24m,male,0


In [ ]:
# ============================================================
# Cell: 生成 celltype_coverage_filtered.csv
# ============================================================
celltype_records = []

for adata, name in datasets_raw:
    obs = adata.obs
    ct_mice = obs.groupby("cell_ontology_class")["mouse.id"].nunique().reset_index()
    ct_mice.columns = ["cell_type", "n_mice"]
    ct_cells = obs["cell_ontology_class"].value_counts().reset_index()
    ct_cells.columns = ["cell_type", "n_cells"]
    
    ct_summary = ct_mice.merge(ct_cells, on="cell_type")
    ct_summary["tissue"] = name
    celltype_records.append(ct_summary)

df_celltype = pd.concat(celltype_records, ignore_index=True)
df_celltype = df_celltype[["tissue", "cell_type", "n_mice", "n_cells"]]
df_celltype = df_celltype.sort_values(["tissue", "n_cells"], ascending=[True, False])

df_celltype.to_csv("../docs/celltype_coverage_filtered.csv", index=False)
print(f"✅ 已保存: ../docs/celltype_coverage_filtered.csv")
print(f"   共 {len(df_celltype)} 条记录")
print(f"\n各组织细胞类型数量:")
df_celltype.groupby("tissue")["cell_type"].count()

✅ 已保存: ../docs/celltype_coverage_filtered.csv
   共 28 条记录

各组织细胞类型数量:


C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\3188652715.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ct_mice = obs.groupby("cell_ontology_class")["mouse.id"].nunique().reset_index()
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\3188652715.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ct_mice = obs.groupby("cell_ontology_class")["mouse.id"].nunique().reset_index()
C:\Users\asd123cheese\AppData\Local\Temp\ipykernel_22480\3188652715.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to

tissue
Fat                 2
Heart_and_Aorta     3
Kidney              3
Liver               3
Lung               13
spleen/marrow       4
Name: cell_type, dtype: int64